# Sólidos semelhantes e troncos

**Módulo 06 — Geometria Espacial** · Notebook 14 de 18

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/06_geometria_espacial/0614a_solidos_semelhantes_e_troncos.ipynb)

**Pré-requisitos**
- Módulo 06 — Geometria Espacial: [`0602a_paralelismo.ipynb`](0602a_paralelismo.ipynb) (Teorema de Tales no espaço), [`0609a_volume_e_principio_de_cavalieri.ipynb`](0609a_volume_e_principio_de_cavalieri.ipynb), [`0610a_piramide.ipynb`](0610a_piramide.ipynb), [`0611a_cilindro.ipynb`](0611a_cilindro.ipynb) e [`0612a_cone.ipynb`](0612a_cone.ipynb)
- Módulo 04 — Geometria Plana: [`0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb`](../04_geometria_plana/0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb) (trapézio), [`0415a_semelhanca_de_triangulos_e_potencia_de_ponto.ipynb`](../04_geometria_plana/0415a_semelhanca_de_triangulos_e_potencia_de_ponto.ipynb) (semelhança) e [`0419a_equivalencia_plana.ipynb`](../04_geometria_plana/0419a_equivalencia_plana.ipynb) (razão entre áreas de figuras semelhantes)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- usar a razão de semelhança para comparar comprimentos, áreas e volumes de sólidos semelhantes;
- descrever a seção de uma pirâmide (ou de um cone) por um plano paralelo à base;
- calcular áreas e volumes de troncos de pirâmide e de cone;
- calcular volumes de troncos de prisma triangular e de cilindro.

**Tempo estimado:** 80 a 100 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_3d.py`, o arquivo com as funções de desenho deste módulo.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/06_geometria_espacial/utils_3d.py"
    urllib.request.urlretrieve(url, "utils_3d.py")

import utils_3d as u3d
from ipywidgets import interact
import ipywidgets as widgets
import math
import numpy as np

## Quanto chocolate tem o coelho gigante?

Numa loja de chocolates, há dois coelhos com exatamente o mesmo formato: um de 10 cm de altura e outro de 20 cm. O pequeno custa R\$ 15,00. O grande, que tem o **dobro** da altura, custa R\$ 80,00. Que absurdo! Ele deveria custar R\$ 30,00, o dobro... certo?

Não! O coelho grande é o dobro do pequeno em **todas** as direções: altura, largura e profundidade. Ele tem $2 \times 2 \times 2 = 8$ vezes mais chocolate. Pelo preço do pequeno, deveria custar R\$ 120,00. O grande é uma pechincha!

🕹️ No widget abaixo, um sólido simples (uma "casinha") é ampliado por um fator $k$. Compare o que acontece com as medidas, a área e o volume.

In [2]:
# A "casinha": um prisma de base quadrada 2 × 2 e altura 2, com uma pirâmide de
# altura 1 em cima (altura total 3). Os vértices 0 a 3 são os cantos do chão, 4 a 7
# os cantos do teto e 8 é a ponta do telhado.
vertices_casinha = [
    (-1, -1, 0), (1, -1, 0), (1, 1, 0), (-1, 1, 0),
    (-1, -1, 2), (1, -1, 2), (1, 1, 2), (-1, 1, 2),
    (0, 0, 3),
]
faces_casinha = [
    [0, 3, 2, 1],                                            # o chão
    [0, 1, 5, 4], [1, 2, 6, 5], [2, 3, 7, 6], [3, 0, 4, 7],  # as quatro paredes
    [4, 5, 8], [5, 6, 8], [6, 7, 8], [7, 4, 8],              # o telhado
]


def ampliar_casinha(k: float) -> None:
    """Desenha a casinha original e a casinha ampliada k vezes e compara as razões."""
    # A ampliação parte do centro do chão da casinha: ela cresce para cima e para os
    # lados, sem sair do chão.
    vertices_ampliados, faces_ampliadas = u3d.escalar_poliedro(
        vertices_casinha, faces_casinha, k
    )

    area_original = u3d.medir_area_total(vertices_casinha, faces_casinha)
    area_ampliada = u3d.medir_area_total(vertices_ampliados, faces_ampliadas)
    volume_original = u3d.medir_volume(vertices_casinha, faces_casinha)
    volume_ampliado = u3d.medir_volume(vertices_ampliados, faces_ampliadas)

    altura_original = 3
    altura_ampliada = k * altura_original
    razao_alturas = altura_ampliada / altura_original
    razao_areas = area_ampliada / area_original         # medidas com medir_area_total
    razao_volumes = volume_ampliado / volume_original   # medidas com medir_volume

    # Só para desenhar ao lado: empurramos a cópia ampliada para a direita.
    deslocamento = 1 + 1.5 + k
    vertices_ao_lado = [(x + deslocamento, y, z) for x, y, z in vertices_ampliados]

    fig = u3d.criar_figura(titulo=f"Casinha original e ampliada com k = {k}")
    fig = u3d.desenhar_poliedro(
        fig, vertices_casinha, faces_casinha, cor=u3d.CORES["plano"]
    )
    fig = u3d.desenhar_poliedro(
        fig, vertices_ao_lado, faces_ampliadas, cor=u3d.CORES["plano_secundario"]
    )
    fig = u3d.adicionar_rotulo(fig, (0, 0, 3.6), "original")
    fig = u3d.adicionar_rotulo(fig, (deslocamento, 0, altura_ampliada + 0.6), f"× {k}")
    fig = u3d.definir_camera(fig, olho=(0.3, -2.0, 0.6))
    u3d.mostrar(fig)

    print(f"{'Medida':<10}{'original':>11}{'ampliada':>11}{'razão':>9}   compare com")
    print(
        f"{'altura':<10}{altura_original:>11.2f}{altura_ampliada:>11.2f}"
        f"{razao_alturas:>9.4f}   k  = {k:.4f}"
    )
    print(
        f"{'área':<10}{area_original:>11.2f}{area_ampliada:>11.2f}"
        f"{razao_areas:>9.4f}   k² = {k**2:.4f}"
    )
    print(
        f"{'volume':<10}{volume_original:>11.2f}{volume_ampliado:>11.2f}"
        f"{razao_volumes:>9.4f}   k³ = {k**3:.4f}"
    )


interact(
    ampliar_casinha,
    k=widgets.FloatSlider(
        min=0.5, max=3, step=0.25, value=2, description="fator k",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='fator k', max=3.0, min=0.5,…

Comprimentos crescem como $k$, áreas como $k^2$ e volumes como $k^3$. Essa regra simples explica o preço dos coelhos de chocolate, o tamanho das maquetes de arquitetura, e até por que não existem formigas do tamanho de cachorros. Ela também é a chave para calcular o volume dos **troncos**: pirâmides e cones com a ponta cortada.

## 1. Sólidos semelhantes

> 📌 **Definição.** Dois sólidos são **semelhantes** quando um é uma ampliação (ou redução) do outro: todos os comprimentos correspondentes estão numa mesma razão $k$, a **razão de semelhança**, e os ângulos correspondentes são iguais.

> 💡 **Dica.** Um jeito de construir um sólido semelhante a outro é a **homotetia**: escolha um ponto fixo $O$ e leve cada ponto $P$ do sólido ao ponto $P'$ da semirreta $OP$ com $OP' = k \cdot OP$. Foi assim que a casinha do widget foi ampliada, com $O$ no centro do chão.

**Teorema**
> Se dois sólidos são semelhantes com razão $k$:
> - a razão entre comprimentos correspondentes (arestas, alturas, raios, diagonais) é $k$;
> - a razão entre **áreas** correspondentes é $k^2$;
> - a razão entre os **volumes** é $k^3$.

**Por quê?** Cada face é uma figura plana ampliada $k$ vezes, e áreas de figuras planas semelhantes estão na razão $k^2$ (módulo 04). Para o volume, pense nos cubinhos do notebook 09: cada cubinho de aresta 1 vira um cubo de aresta $k$, com volume $k^3$.

> ⚠️ **Erro comum.** Achar que "o dobro do tamanho" significa "o dobro de tudo". Dobrando as medidas, a área fica 4 vezes maior e o volume 8 vezes maior.

**Aplicações**
- **Maquetes:** numa maquete na escala 1:100, cada comprimento real é 100 vezes o da maquete; cada área real é $100^2 = 10\,000$ vezes a da maquete; e cada volume real é $100^3 = 1\,000\,000$ de vezes o da maquete.
- **Embalagens:** uma lata com o dobro das medidas leva 8 vezes mais produto, mas gasta só 4 vezes mais material.

> 🕵️ **Curiosidade histórica.** Em 1638, **Galileu Galilei** usou essa ideia (hoje chamada de **lei do quadrado e do cubo**) para explicar por que não existem gigantes com as mesmas proporções dos humanos. Se um animal fosse ampliado 10 vezes, seu peso (que depende do volume) ficaria 1000 vezes maior, mas a força dos seus ossos (que depende da área da seção dos ossos) ficaria só 100 vezes maior. Os ossos não aguentariam! É por isso que animais grandes, como os elefantes, têm pernas proporcionalmente muito mais grossas que animais pequenos. E é por isso que o King Kong não poderia existir.

Veja a "lei de Galileu" em números.

In [3]:
print(f"{'ampliação':>10}{'peso':>10}{'área dos ossos':>17}{'pressão nos ossos':>20}")
for k in [1, 2, 5, 10, 20]:
    peso = k**3                # o peso acompanha o volume
    area_dos_ossos = k**2      # a seção dos ossos é uma área
    pressao_nos_ossos = peso / area_dos_ossos   # k³ / k² = k
    print(f"{k:>10}{peso:>10}{area_dos_ossos:>17}{pressao_nos_ossos:>20.0f}")

# As três curvas: comprimento (k), área (k²) e volume (k³).
ks = np.linspace(1, 5, 81)
fig = u3d.criar_grafico(
    titulo="Quantas vezes maior fica cada medida?",
    rotulo_x="fator de ampliação k",
    rotulo_y="quantas vezes maior",
)
fig = u3d.desenhar_curva(fig, ks, ks, rotulo="comprimentos: k", cor=u3d.CORES["plano"])
fig = u3d.desenhar_curva(
    fig, ks, ks**2, rotulo="áreas: k²", cor=u3d.CORES["plano_secundario"]
)
fig = u3d.desenhar_curva(fig, ks, ks**3, rotulo="volumes: k³", cor=u3d.CORES["destaque"])
u3d.mostrar(fig)

 ampliação      peso   área dos ossos   pressão nos ossos
         1         1                1                   1
         2         8                4                   2
         5       125               25                   5
        10      1000              100                  10
        20      8000              400                  20


**Pare e pense**

1. Dois cubos têm arestas 2 cm e 6 cm. Qual é a razão entre as áreas? E entre os volumes?
2. Uma maquete na escala 1:50 tem uma piscina com 40 mL de água. Quantos litros tem a piscina real?

<details>
<summary>Ver respostas</summary>

1. $k = 3$: áreas na razão $9$; volumes na razão $27$.
2. $40 \cdot 50^3 = 5\,000\,000$ mL $= 5000$ L.
</details>

## 2. Seção de uma pirâmide (ou de um cone) por um plano paralelo à base

No notebook 10, você viu que a seção de uma pirâmide de altura $H$, feita por um plano paralelo à base a uma distância $d$ do vértice, tem área $S_b \left(\frac{d}{H}\right)^2$. Agora podemos dizer mais:

**Teorema**
> O plano paralelo à base separa a pirâmide em duas partes. A parte de cima é uma **pirâmide semelhante** à original, com razão de semelhança $k = \dfrac{d}{H}$. Portanto:
> - a seção é um polígono semelhante à base, com $\dfrac{\text{área da seção}}{\text{área da base}} = \left(\dfrac{d}{H}\right)^2$;
> - $\dfrac{\text{volume da pirâmide pequena}}{\text{volume da pirâmide grande}} = \left(\dfrac{d}{H}\right)^3$.

A justificativa usa o Teorema de Tales no espaço (notebook 02): o plano da seção e o plano da base são paralelos, então todas as arestas laterais ficam divididas na mesma razão $\frac{d}{H}$.

O mesmo vale para o **cone**: a parte de cima é um cone semelhante, com raio $r = R \cdot \frac{d}{H}$.

A parte de **baixo**, que sobra quando retiramos a pirâmide pequena, é o **tronco**.

🕹️ Mova o plano de corte e separe as duas partes.

In [4]:
def cortar_paralelo_a_base(solido: str, posicao_do_corte: float, afastamento: float) -> None:
    """Corta a pirâmide (ou o cone) por um plano paralelo à base e separa as partes."""
    k = posicao_do_corte    # k = d / H: a fração da altura medida a partir do vértice
    razao_areas = k**2
    razao_volumes = k**3
    fracao_do_tronco = 1 - k**3

    fig = u3d.criar_figura(titulo=f"{solido} cortada(o) com d/H = {k:.1f}")
    if solido == "Pirâmide":
        vertices, faces = u3d.piramide_regular(4, 4, 6)
        base, vertice = vertices[:4], vertices[4]
        # O plano fica a uma distância k·H do VÉRTICE, ou seja, a uma fração 1 − k da
        # altura medida a partir da BASE.
        tronco = u3d.tronco_de_piramide(base, vertice, 1 - k)
        secao = tronco[0][4:]   # a base menor do tronco é a seção
        pequena = u3d.piramide(secao, vertice)
        pequena_afastada, tronco_afastado = u3d.afastar_pecas(
            [pequena, tronco], afastamento
        )
        fig = u3d.desenhar_poliedro(fig, *tronco_afastado, cor=u3d.CORES["plano"])
        fig = u3d.desenhar_poliedro(fig, *pequena_afastada, cor=u3d.CORES["destaque"])
    else:
        raio, altura = 2, 6
        # O cone pequeno tem raio k·R e altura k·H; o tronco fica com altura (1 − k)·H.
        altura_do_tronco = (1 - k) * altura
        fig = u3d.desenhar_tronco_de_cone(
            fig, (0, 0, 0), raio, k * raio, altura_do_tronco, cor=u3d.CORES["plano"]
        )
        # Só para desenhar: o cone pequeno sobe `afastamento` unidades.
        fig = u3d.desenhar_cone(
            fig, (0, 0, altura + afastamento), (0, 0, altura_do_tronco + afastamento),
            k * raio, cor=u3d.CORES["destaque"],
        )
    fig = u3d.definir_camera(fig, olho=(1.5, -1.5, 0.7))
    u3d.mostrar(fig)

    print(f"k = d/H = {k:.2f}")
    print(f"Área da seção / área da base = k² = {razao_areas:.4f}")
    print(f"Volume da parte de cima / volume do sólido inteiro = k³ = {razao_volumes:.4f}")
    print(f"Fração do volume que fica no tronco = 1 − k³ = {fracao_do_tronco:.4f}")
    if solido == "Pirâmide":
        conferencia_areas = u3d.medir_area_do_poligono(secao) / u3d.medir_area_do_poligono(
            base
        )
        conferencia_volumes = u3d.medir_volume(*pequena) / u3d.medir_volume(
            vertices, faces
        )
        print(f"Conferência com o utils_3d: áreas {conferencia_areas:.4f}, "
              f"volumes {conferencia_volumes:.4f}")


interact(
    cortar_paralelo_a_base,
    solido=widgets.RadioButtons(options=["Pirâmide", "Cone"], value="Pirâmide"),
    posicao_do_corte=widgets.FloatSlider(
        min=0.1, max=0.9, step=0.1, value=0.5, description="corte d/H",
        continuous_update=False,
    ),
    afastamento=widgets.FloatSlider(
        min=0, max=2, step=0.5, value=1, description="afastamento",
        continuous_update=False,
    ),
);

interactive(children=(RadioButtons(description='solido', options=('Pirâmide', 'Cone'), value='Pirâmide'), Floa…

**Pare e pense**

1. Uma pirâmide de altura 12 é cortada por um plano paralelo à base, a 4 do vértice. Que fração do volume fica na pirâmide pequena?
2. Onde cortar uma pirâmide (a que fração da altura, a partir do vértice) para que a pirâmide pequena tenha metade do volume?

<details>
<summary>Ver respostas</summary>

1. $k = \frac{4}{12} = \frac{1}{3}$, então a pirâmide pequena tem $\left(\frac{1}{3}\right)^3 = \frac{1}{27}$ do volume.
2. $k^3 = \frac{1}{2}$, então $k = \sqrt[3]{0{,}5} \approx 0{,}794$: a cerca de 79% da altura, a partir do vértice. (Bem mais perto da base do que a intuição diz!)
</details>

## 3. Tronco de pirâmide de bases paralelas

> 📌 **Definição.** O **tronco de pirâmide** (de bases paralelas) é a parte de uma pirâmide compreendida entre a base e uma seção paralela à base.

**Elementos:** a **base maior** (área $B$), a **base menor** (área $b$), a **altura** $h$ (distância entre os planos das bases) e as **faces laterais**, que são **trapézios**. No tronco de pirâmide **regular**, as faces laterais são trapézios isósceles congruentes, e a altura de cada trapézio é o **apótema do tronco** $g_t$.

### Volume

O tronco é a pirâmide grande menos a pirâmide pequena. Seja $H$ a altura da pirâmide grande; a pequena tem altura $H - h$. Pela semelhança:
$$\frac{b}{B} = \left(\frac{H - h}{H}\right)^2$$

Fazendo as contas (isolando $H$ e substituindo em $\frac{1}{3}BH - \frac{1}{3}b(H - h)$), chega-se a:
$$\boxed{\text{volume} = \frac{h}{3}\left(B + \sqrt{B\,b} + b\right)}$$

<details>
<summary>Ver as contas</summary>

Da semelhança, $\sqrt{\frac{b}{B}} = \frac{H - h}{H}$, ou seja, $H\sqrt{b} = (H - h)\sqrt{B}$. Isolando: $H = \dfrac{h\sqrt{B}}{\sqrt{B} - \sqrt{b}}$ e $H - h = \dfrac{h\sqrt{b}}{\sqrt{B} - \sqrt{b}}$.

$$\text{volume} = \frac{1}{3}\left(B H - b(H - h)\right) = \frac{h}{3} \cdot \frac{B\sqrt{B} - b\sqrt{b}}{\sqrt{B} - \sqrt{b}}$$

Usando $x^3 - y^3 = (x - y)(x^2 + xy + y^2)$ com $x = \sqrt{B}$ e $y = \sqrt{b}$:
$$\frac{B\sqrt{B} - b\sqrt{b}}{\sqrt{B} - \sqrt{b}} = B + \sqrt{Bb} + b$$
</details>

### Área lateral (tronco regular)

Num tronco de pirâmide regular com bases de $n$ lados ($\ell$ na base maior, $\ell'$ na menor), cada face lateral é um trapézio de área $\frac{(\ell + \ell')\,g_t}{2}$. Somando as $n$ faces:
$$S_\ell = (p + p')\, g_t$$
em que $p$ e $p'$ são os semiperímetros das bases. O apótema do tronco sai de um trapézio retângulo "dentro" do tronco: $g_t^2 = h^2 + (m - m')^2$, com $m$ e $m'$ os apótemas das bases.

> 🕵️ **Curiosidade histórica.** O **Papiro de Moscou**, um documento egípcio de cerca de 1850 a.C., contém um problema resolvido que calcula corretamente o volume de um tronco de pirâmide de bases quadradas, com a mesma fórmula que você acabou de ver. É um dos resultados mais antigos da Geometria Espacial de que se tem registro.

**No dia a dia:** a base de um pilar, um vaso de base quadrada, uma lixeira, pirâmides "achatadas" de algumas construções antigas.

🕹️ Monte um tronco de pirâmide regular.

In [ ]:
def montar_tronco_de_piramide(
    lados: int, lado_maior: float, lado_menor: float, altura: float,
    mostrar_piramide: bool,
) -> None:
    """Desenha um tronco de pirâmide regular e calcula as suas medidas."""
    if lado_menor >= lado_maior:
        print("O lado menor precisa ser menor que o lado maior. Diminua o lado menor.")
        return

    # Apótemas das bases (módulo 04): m = ℓ / (2 · tg(180° / n)).
    m_maior = lado_maior / (2 * math.tan(math.pi / lados))
    m_menor = lado_menor / (2 * math.tan(math.pi / lados))
    # Área de um polígono regular: semiperímetro × apótema = n · ℓ · m / 2.
    B = lados * lado_maior * m_maior / 2
    b = lados * lado_menor * m_menor / 2
    volume = altura / 3 * (B + math.sqrt(B * b) + b)
    g_t = math.sqrt(altura**2 + (m_maior - m_menor)**2)
    area_lateral = (lados * lado_maior / 2 + lados * lado_menor / 2) * g_t
    area_total = area_lateral + B + b
    H = altura * lado_maior / (lado_maior - lado_menor)    # altura da pirâmide completa (semelhança)

    tronco = u3d.tronco_de_piramide_regular(lados, lado_maior, lado_menor, altura)
    fig = u3d.criar_figura(titulo=f"Tronco de pirâmide regular de base com {lados} lados")
    fig = u3d.desenhar_poliedro(fig, *tronco, cor=u3d.CORES["plano"])
    piramide_cabe = H <= 4 * altura
    if mostrar_piramide and piramide_cabe:
        # A pirâmide pequena que foi retirada: a base dela é a base menor do tronco.
        base_menor = tronco[0][lados:]
        pequena = u3d.piramide(base_menor, (0, 0, H))
        fig = u3d.desenhar_poliedro(fig, *pequena, cor=u3d.CORES["apoio"], opacidade=0.15)
    fig = u3d.definir_camera(fig, olho=(1.5, -1.5, 0.7))
    u3d.mostrar(fig)

    print(f"Área da base maior: B = {B:.4f}")
    print(f"Área da base menor: b = {b:.4f}")
    print(f"Volume = h/3 · (B + √(Bb) + b) = {volume:.4f}")
    print(f"Apótema do tronco: g_t = √(h² + (m − m')²) = {g_t:.4f}")
    print(f"Área lateral = (p + p') · g_t = {area_lateral:.4f}")
    print(f"Área total = área lateral + B + b = {area_total:.4f}")
    print(f"Altura da pirâmide completa: H = {H:.4f}")
    if mostrar_piramide and not piramide_cabe:
        print("(A pirâmide completa é alta demais para caber na figura: as bases têm "
              "lados quase iguais.)")
    print(f"Conferência com o utils_3d: volume {u3d.medir_volume(*tronco):.4f}, "
          f"área total {u3d.medir_area_total(*tronco):.4f}")


interact(
    montar_tronco_de_piramide,
    lados=widgets.Dropdown(options=[3, 4, 6], value=4, description="lados da base"),
    lado_maior=widgets.FloatSlider(
        min=2, max=8, step=0.5, value=6, description="lado maior ℓ",
        continuous_update=False,
    ),
    lado_menor=widgets.FloatSlider(
        min=0.5, max=7.5, step=0.5, value=2, description="lado menor ℓ'",
        continuous_update=False,
    ),
    altura=widgets.FloatSlider(
        min=1, max=8, step=0.5, value=4, description="altura h",
        continuous_update=False,
    ),
    mostrar_piramide=widgets.Checkbox(
        value=True, description="Mostrar a pirâmide completa"
    ),
);

interactive(children=(Dropdown(description='lados da base', index=1, options=(3, 4, 6), value=4), FloatSlider(…

**Pare e pense**

1. Um tronco de pirâmide quadrangular regular tem bases de lados 6 e 3 e altura 4. Qual é o volume?
2. Por que a fórmula do volume do tronco não é simplesmente "média das bases × altura"?

<details>
<summary>Ver respostas</summary>

1. $B = 36$, $b = 9$: volume $= \frac{4}{3}(36 + 18 + 9) = 84$.
2. Porque as seções do tronco não variam "em linha reta" de $B$ até $b$: a área de cada seção depende do **quadrado** da distância ao vértice. A média $\frac{B + b}{2} \cdot h$ daria $90$, mais do que os $84$ verdadeiros.
</details>

## 4. Tronco de cone de bases paralelas

> 📌 **Definição.** O **tronco de cone** (de bases paralelas) é a parte de um cone compreendida entre a base e uma seção paralela à base.

Vamos trabalhar com o tronco de cone **reto**: raio maior $R$, raio menor $r$, altura $h$ e geratriz $g$. Na seção meridiana, o tronco é um **trapézio isósceles**, e a geratriz sai de Pitágoras:
$$g^2 = h^2 + (R - r)^2$$

### Volume

É a mesma ideia do tronco de pirâmide, com $B = \pi R^2$ e $b = \pi r^2$ (então $\sqrt{Bb} = \pi R r$):
$$\boxed{\text{volume} = \frac{\pi h}{3}\left(R^2 + R r + r^2\right)}$$

### Área lateral

A planificação da superfície lateral é um **setor de coroa circular**: o setor do cone grande menos o setor do cone pequeno. Fazendo as contas, chega-se a uma fórmula parecida com a do tronco de pirâmide:
$$\boxed{S_\ell = \pi (R + r)\, g} \qquad\qquad S_t = \pi(R + r)\,g + \pi R^2 + \pi r^2$$

> ⚠️ **Erro comum.** Calcular o volume do tronco como "área média das bases × altura", ou seja, $\pi \frac{R^2 + r^2}{2} h$. Esse valor é **sempre maior** que o verdadeiro (a não ser que $R = r$).

**No dia a dia:** copos, baldes, vasos de planta, abajures, potes de pipoca, a casquinha de sorvete com a ponta cortada.

🕹️ Monte o tronco e veja a planificação.

In [ ]:
def explorar_tronco_de_cone(
    raio_maior: float, raio_menor: float, altura: float, vista: str
) -> None:
    """Desenha o tronco de cone (ou a sua planificação) e calcula as suas medidas."""
    if raio_menor >= raio_maior:
        print("O raio menor precisa ser menor que o raio maior. Diminua o raio menor.")
        return

    g = math.sqrt(altura**2 + (raio_maior - raio_menor)**2)
    volume = math.pi * altura / 3 * (raio_maior**2 + raio_maior * raio_menor + raio_menor**2)
    volume_errado = math.pi * (raio_maior**2 + raio_menor**2) / 2 * altura
    area_lateral = math.pi * (raio_maior + raio_menor) * g
    area_total = area_lateral + math.pi * raio_maior**2 + math.pi * raio_menor**2

    H = altura * raio_maior / (raio_maior - raio_menor)    # altura do cone completo
    if vista == "Tronco":
        fig = u3d.criar_figura(titulo="Tronco de cone (e o cone pequeno retirado)")
        fig = u3d.desenhar_tronco_de_cone(
            fig, (0, 0, 0), raio_maior, raio_menor, altura, mostrar_eixo=True,
            mostrar_geratrizes=8, mostrar_cone_completo=True,
        )
        fig = u3d.definir_camera(fig, olho=(1.5, -1.5, 0.7))
    else:
        G = math.sqrt(H**2 + raio_maior**2)                    # geratriz do cone completo
        G_pequeno = G * raio_menor / raio_maior                # geratriz do cone retirado
        angulo = 360 * raio_maior / G
        fig = u3d.criar_figura(titulo=f"Planificação: setor de coroa de {angulo:.1f}°")
        fig = u3d.desenhar_setor_de_coroa(fig, G_pequeno, G, angulo)
        # As duas bases, ao lado do setor de coroa.
        x_maior = G + 1 + raio_maior
        x_menor = x_maior + raio_maior + 1 + raio_menor
        for x, raio in [(x_maior, raio_maior), (x_menor, raio_menor)]:
            disco = u3d.poligono_regular(60, raio=raio, centro=(x, 0, 0))
            fig = u3d.desenhar_poligono(fig, disco, cor=u3d.CORES["plano"], opacidade=0.6)
        fig = u3d.definir_camera(fig, olho=(0, -0.4, 2.2))
        print(f"Raios do setor de coroa: G' = {G_pequeno:.4f} e G = {G:.4f}; "
              f"ângulo = 360° · R / G = {angulo:.2f}°")
    u3d.mostrar(fig)

    print(f"Geratriz: g = √(h² + (R − r)²) = {g:.4f}")
    print(f"Volume = πh/3 · (R² + Rr + r²) = {volume:.4f}")
    print(f"\"Volume errado\" (média das bases × altura) = {volume_errado:.4f}   "
          f"(sobra {volume_errado - volume:.4f})")
    print(f"Área lateral = π(R + r)g = {area_lateral:.4f}")
    print(f"Área total = {area_total:.4f}")
    # Conferência: um tronco de pirâmide regular de 2000 lados quase não se distingue
    # do tronco de cone.
    quase_cone = u3d.tronco_de_piramide(
        u3d.poligono_regular(2000, raio=raio_maior), (0, 0, H), altura / H
    )
    print(f"Conferência (tronco de pirâmide de 2000 lados): volume "
          f"{u3d.medir_volume(*quase_cone):.4f}")


interact(
    explorar_tronco_de_cone,
    raio_maior=widgets.FloatSlider(
        min=1, max=5, step=0.25, value=3, description="raio maior R",
        continuous_update=False,
    ),
    raio_menor=widgets.FloatSlider(
        min=0.25, max=4.75, step=0.25, value=1, description="raio menor r",
        continuous_update=False,
    ),
    altura=widgets.FloatSlider(
        min=1, max=8, step=0.5, value=4, description="altura h",
        continuous_update=False,
    ),
    vista=widgets.RadioButtons(options=["Tronco", "Planificação"], value="Tronco"),
);

interactive(children=(FloatSlider(value=3.0, continuous_update=False, description='raio maior R', max=5.0, min…

**Pare e pense**

1. Um copo tem a forma de tronco de cone, com 4 cm de raio na boca, 3 cm de raio no fundo e 10 cm de altura. Quantos mL ele comporta?
2. O que acontece com as fórmulas do tronco de cone quando $r = 0$? E quando $r = R$?

<details>
<summary>Ver respostas</summary>

1. $\frac{\pi \cdot 10}{3}(16 + 12 + 9) = \frac{370\pi}{3} \approx 387{,}5$ cm³, ou seja, cerca de 387,5 mL.
2. Com $r = 0$, viram as fórmulas do **cone** ($\frac{1}{3}\pi R^2 h$ e $\pi R g$). Com $r = R$, viram as do **cilindro** ($\pi R^2 h$ e $2\pi R h$).
</details>

## 5. Tronco de prisma triangular e tronco de cilindro

Até aqui, os troncos foram cortados por planos **paralelos** à base. Agora vamos cortar **prismas** e **cilindros** por um plano **inclinado**.

### Tronco de prisma triangular

> 📌 **Definição.** Um **tronco de prisma triangular** é a parte de um prisma triangular compreendida entre a base e um plano que corta todas as arestas laterais, sem ser paralelo à base.

As três arestas laterais continuam paralelas, mas agora têm comprimentos diferentes: $a$, $b$ e $c$. Se as arestas laterais forem perpendiculares à base (de área $S$):
$$\boxed{\text{volume} = S \cdot \frac{a + b + c}{3}}$$

Ou seja: é como um prisma cuja altura é a **média** das três arestas laterais. A demonstração divide o tronco em **três pirâmides**, como no notebook 10.

### Tronco de cilindro

> 📌 **Definição.** Um **tronco de cilindro** é a parte de um cilindro reto compreendida entre a base e um plano inclinado que corta todas as geratrizes. A base de cima é uma **elipse**.

Sejam $g_M$ e $g_m$ a maior e a menor geratriz. Pegue **dois** troncos iguais e encaixe um de cabeça para baixo sobre o outro: eles formam um cilindro completo de altura $g_M + g_m$. Então cada tronco tem metade desse volume:
$$\boxed{\text{volume} = \pi r^2 \cdot \frac{g_M + g_m}{2}} \qquad\qquad S_\ell = 2\pi r \cdot \frac{g_M + g_m}{2}$$

A média $\frac{g_M + g_m}{2}$ é exatamente o comprimento do eixo do tronco (do centro da base ao centro da elipse).

**No dia a dia:** canos cortados "em bisel", o tubo de uma chaminé que atravessa um telhado inclinado, troncos de árvore cortados na diagonal, blocos de concreto com o topo inclinado.

🕹️ Escolha um dos sólidos. (Os controles $a$, $b$ e $c$ valem para o tronco de prisma; o raio e as geratrizes, para os troncos de cilindro.)

In [ ]:
def cortar_inclinado(
    solido: str, a: float, b: float, c: float,
    raio: float, geratriz_menor: float, geratriz_maior: float,
) -> None:
    """Desenha um tronco de prisma triangular ou de cilindro e calcula o volume."""
    if solido == "Tronco de prisma triangular":
        # Base: o triângulo retângulo de catetos 4 e 3, com área 4 · 3 / 2 = 6.
        base = [(0, 0, 0), (4, 0, 0), (0, 3, 0)]
        area_da_base = 4 * 3 / 2
        volume = area_da_base * (a + b + c) / 3
        tronco = u3d.tronco_de_prisma(base, [a, b, c])

        fig = u3d.criar_figura(titulo="Tronco de prisma triangular reto")
        fig = u3d.desenhar_poliedro(fig, *tronco, cor=u3d.CORES["plano"])
        # O nome de cada aresta lateral, ao lado do seu ponto médio.
        for (x, y, _), comprimento, nome in zip(base, [a, b, c], "abc"):
            fig = u3d.adicionar_rotulo(fig, (x - 0.4, y - 0.4, comprimento / 2), nome)
        fig = u3d.definir_camera(fig, olho=(1.4, -1.6, 0.8))
        u3d.mostrar(fig)

        print(f"Área da base: S = {area_da_base:g}")
        print(f"Volume = S · (a + b + c) / 3 = {area_da_base:g} · {a + b + c:g} / 3 = {volume:.4f}")
        print(f"Conferência com o utils_3d: {u3d.medir_volume(*tronco):.4f}")
        return

    if geratriz_menor > geratriz_maior:
        print("A geratriz menor não pode passar da maior. Ajuste os controles.")
        return
    media = (geratriz_maior + geratriz_menor) / 2   # o comprimento do eixo do tronco
    volume = math.pi * raio**2 * (geratriz_maior + geratriz_menor) / 2
    area_lateral = 2 * math.pi * raio * (geratriz_maior + geratriz_menor) / 2

    if solido == "Tronco de cilindro":
        fig = u3d.criar_figura(titulo="Tronco de cilindro")
        fig = u3d.desenhar_tronco_de_cilindro(
            fig, raio, geratriz_menor, geratriz_maior, cor=u3d.CORES["plano"],
            mostrar_eixo=True,
        )
        fig = u3d.definir_camera(fig, olho=(0.4, -2.0, 0.6))
        u3d.mostrar(fig)
        print(f"Comprimento do eixo: (g_M + g_m) / 2 = {media:.4f}")
        print(f"Volume = πr² · (g_M + g_m) / 2 = {volume:.4f}")
        print(f"Área lateral = 2πr · (g_M + g_m) / 2 = {area_lateral:.4f}")
    else:
        altura_do_cilindro = geratriz_maior + geratriz_menor
        volume_do_cilindro = math.pi * raio**2 * altura_do_cilindro
        fig = u3d.criar_figura(titulo="Dois troncos iguais formam um cilindro")
        fig = u3d.desenhar_tronco_de_cilindro(
            fig, raio, geratriz_menor, geratriz_maior, cor=u3d.CORES["plano"]
        )
        # Só para desenhar: uma cópia do tronco virada de cabeça para baixo e girada
        # 180°, encaixada em cima do primeiro.
        fig = u3d.desenhar_tronco_de_cilindro(
            fig, raio, geratriz_menor, geratriz_maior,
            cor=u3d.CORES["plano_secundario"], virado=True,
        )
        # O contorno do cilindro completo.
        fig = u3d.desenhar_cilindro(
            fig, (0, 0, 0), raio * 1.03, (0, 0, altura_do_cilindro),
            cor=u3d.CORES["apoio"], opacidade=0.06, mostrar_bases=False,
        )
        fig = u3d.definir_camera(fig, olho=(0.4, -2.0, 0.6))
        u3d.mostrar(fig)
        print(f"Altura do cilindro completo: g_M + g_m = {altura_do_cilindro:g}")
        print(f"Volume do cilindro completo: πr² · (g_M + g_m) = {volume_do_cilindro:.4f}")
        print(f"Volume de cada tronco (a metade): {volume_do_cilindro / 2:.4f}")
        print(f"Pela fórmula: πr² · (g_M + g_m) / 2 = {volume:.4f}")


interact(
    cortar_inclinado,
    solido=widgets.RadioButtons(
        options=[
            "Tronco de prisma triangular",
            "Tronco de cilindro",
            "Dois troncos de cilindro formam um cilindro",
        ],
        value="Tronco de prisma triangular",
        layout=widgets.Layout(width="max-content"),
    ),
    a=widgets.FloatSlider(
        min=1, max=8, step=0.5, value=3, description="aresta a",
        continuous_update=False,
    ),
    b=widgets.FloatSlider(
        min=1, max=8, step=0.5, value=4, description="aresta b",
        continuous_update=False,
    ),
    c=widgets.FloatSlider(
        min=1, max=8, step=0.5, value=8, description="aresta c",
        continuous_update=False,
    ),
    raio=widgets.FloatSlider(
        min=1, max=3, step=0.25, value=2, description="raio r",
        continuous_update=False,
    ),
    geratriz_menor=widgets.FloatSlider(
        min=1, max=10, step=0.5, value=4, description="geratriz g_m",
        continuous_update=False,
    ),
    geratriz_maior=widgets.FloatSlider(
        min=1, max=10, step=0.5, value=8, description="geratriz g_M",
        continuous_update=False,
    ),
);

interactive(children=(RadioButtons(description='solido', layout=Layout(width='max-content'), options=('Tronco …

**Pare e pense**

1. Um cano de raio 2 cm foi cortado em bisel: a maior geratriz mede 9 cm e a menor, 5 cm. Qual é o volume desse pedaço de cano (pensando nele como um cilindro maciço)?
2. Um tronco de prisma triangular reto tem base de área 6 e arestas laterais 2, 3 e 7. Qual é o volume?

<details>
<summary>Ver respostas</summary>

1. $\pi \cdot 4 \cdot \frac{9 + 5}{2} = 28\pi \approx 87{,}96$ cm³.
2. $6 \cdot \frac{2 + 3 + 7}{3} = 24$.
</details>

## Resumo

**Sólidos semelhantes** (razão $k$): comprimentos $\times k$; áreas $\times k^2$; volumes $\times k^3$.

**Seção paralela à base** (a distância $d$ do vértice, altura total $H$): a parte de cima é semelhante, com $k = \frac{d}{H}$. Áreas: $k^2$; volumes: $k^3$.

**Tronco de pirâmide:** volume $= \dfrac{h}{3}\left(B + \sqrt{Bb} + b\right)$. Regular: $S_\ell = (p + p')\,g_t$, com $g_t^2 = h^2 + (m - m')^2$.

**Tronco de cone reto:** $g^2 = h^2 + (R - r)^2$; volume $= \dfrac{\pi h}{3}\left(R^2 + Rr + r^2\right)$; $S_\ell = \pi(R + r)g$. Planificação lateral: setor de coroa circular.

⚠️ O volume de um tronco **não** é "média das bases × altura".

**Tronco de prisma triangular reto:** volume $= S \cdot \dfrac{a + b + c}{3}$.

**Tronco de cilindro:** volume $= \pi r^2 \cdot \dfrac{g_M + g_m}{2}$; $S_\ell = 2\pi r \cdot \dfrac{g_M + g_m}{2}$.

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**. Use `math.pi` nos cálculos.

O gabarito comentado está em [`0614b_solidos_semelhantes_e_troncos_gabarito.ipynb`](0614b_solidos_semelhantes_e_troncos_gabarito.ipynb). Tente sozinho antes de olhar!

### Básico

In [ ]:
# Exercício Básico 1: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) Dobrando todas as medidas de um sólido, a área da superfície fica multiplicada
#     por 4.
# (b) Dobrando todas as medidas de um sólido, o volume fica multiplicado por 6.
# (c) A seção de uma pirâmide por um plano paralelo à base é semelhante à base.
# (d) O volume de um tronco de pirâmide é a média das áreas das bases vezes a altura.
# (e) As faces laterais de um tronco de pirâmide regular são trapézios isósceles.
respostas_ex1 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
    "e": ...,
}

print(respostas_ex1)

In [ ]:
# Verificação
gabarito_ex1 = {"a": True, "b": False, "c": True, "d": False, "e": True}
dicas_ex1 = {
    "a": "as áreas ficam multiplicadas por k² (seção 1 - Sólidos semelhantes)",
    "b": "os volumes ficam multiplicados por k³, e 2³ = 8 (seção 1 - Sólidos semelhantes)",
    "c": "a parte de cima é uma pirâmide semelhante à original (seção 2 - Seção de uma pirâmide (ou de um cone) por um plano paralelo à base)",
    "d": "o volume é h/3 · (B + √(Bb) + b) (seção 3 - Tronco de pirâmide de bases paralelas)",
    "e": "as faces laterais do tronco regular (seção 3 - Tronco de pirâmide de bases paralelas)",
}
for item, certa in gabarito_ex1.items():
    assert respostas_ex1[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo; {dicas_ex1[item]}."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: razões de semelhança.
# (a) Dois sólidos são semelhantes com razão k = 3. Qual é a razão entre as áreas?
# (b) E entre os volumes?
# (c) Uma maquete está na escala 1:100. Qual é a razão entre uma área real e a área
#     correspondente na maquete?
# (d) E entre os volumes?
razao_areas_ex2 = ...
razao_volumes_ex2 = ...
razao_areas_maquete_ex2 = ...
razao_volumes_maquete_ex2 = ...

print(razao_areas_ex2, razao_volumes_ex2, razao_areas_maquete_ex2, razao_volumes_maquete_ex2)

In [ ]:
# Verificação
respostas_ex2 = [
    ("a", razao_areas_ex2, 9),
    ("b", razao_volumes_ex2, 27),
    ("c", razao_areas_maquete_ex2, 10_000),
    ("d", razao_volumes_maquete_ex2, 1_000_000),
]
for item, resposta, certa in respostas_ex2:
    assert resposta is not ... and abs(resposta - certa) <= 0.001, (
        f"Tente de novo, revise: no item ({item}), áreas: k²; volumes: k³ "
        "(seção 1 - Sólidos semelhantes)."
    )
print("Certo!")

In [ ]:
# Exercício Básico 3: a pirâmide cortada. Uma pirâmide de altura 12 e volume 216 é
# cortada por um plano paralelo à base, a 4 do vértice. Calcule:
# (a) o volume da pirâmide pequena (a de cima);
# (b) o volume do tronco (a parte de baixo).
volume_pequena_ex3 = ...
volume_tronco_ex3 = ...

print(volume_pequena_ex3, volume_tronco_ex3)

In [ ]:
# Verificação
respostas_ex3 = [("a", volume_pequena_ex3, 8), ("b", volume_tronco_ex3, 208)]
for item, resposta, certa in respostas_ex3:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), k = 4/12 = 1/3; o volume da pequena "
        "é 216 · k³, e o tronco é o que sobra (seção 2 - Seção de uma pirâmide (ou de um cone) por um plano paralelo à base)."
    )
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: tronco de pirâmide quadrangular regular. As bases têm
# lados 6 e 3, e a altura é 4. Calcule:
# (a) o volume;
# (b) o apótema do tronco g_t;
# (c) a área lateral;
# (d) a área total.
volume_ex4 = ...
apotema_ex4 = ...
area_lateral_ex4 = ...
area_total_ex4 = ...

print(volume_ex4, apotema_ex4, area_lateral_ex4, area_total_ex4)

In [ ]:
# Verificação
g_t = math.sqrt(18.25)
respostas_ex4 = [
    ("a", volume_ex4, 84, "volume = h/3 · (B + √(Bb) + b), com B = 36 e b = 9"),
    ("b", apotema_ex4, g_t, "g_t² = h² + (m − m')², com m = 3 e m' = 1,5"),
    ("c", area_lateral_ex4, 18 * g_t, "S_l = (p + p') · g_t, com p = 12 e p' = 6"),
    ("d", area_total_ex4, 18 * g_t + 45, "área total = área lateral + B + b"),
]
for item, resposta, certa, dica in respostas_ex4:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), {dica} (seção 3 - Tronco de pirâmide de bases paralelas)."
    )
print("Certo!")

In [ ]:
# Exercício Intermediário 5: o copo. Um copo em forma de tronco de cone tem raio 4 cm
# na boca, 3 cm no fundo e altura 10 cm. Calcule:
# (a) o volume, em mL (1 cm³ = 1 mL);
# (b) o "volume errado" que seria obtido com a média das áreas das bases vezes a
#     altura.
volume_ex5 = ...
volume_errado_ex5 = ...

print(volume_ex5, volume_errado_ex5)

In [ ]:
# Verificação
assert volume_ex5 is not ... and abs(volume_ex5 - 370 * math.pi / 3) <= 0.01, (
    "Tente de novo, revise: no item (a), volume = πh/3 · (R² + Rr + r²) "
    "(seção 4 - Tronco de cone de bases paralelas)."
)
assert volume_errado_ex5 is not ... and abs(volume_errado_ex5 - 125 * math.pi) <= 0.01, (
    "Tente de novo, revise: no item (b), a média das áreas das bases é "
    "(πR² + πr²) / 2; multiplique pela altura (seção 4 - Tronco de cone de bases paralelas)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 6: o cano em bisel. Um cilindro de raio 2 cm é cortado em
# bisel, com geratrizes de 5 cm e 9 cm. Calcule:
# (a) o volume;
# (b) a área lateral.
volume_ex6 = ...
area_lateral_ex6 = ...

print(volume_ex6, area_lateral_ex6)

In [ ]:
# Verificação
respostas_ex6 = [("a", volume_ex6, 28 * math.pi), ("b", area_lateral_ex6, 28 * math.pi)]
for item, resposta, certa in respostas_ex6:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), use a média das geratrizes como se "
        "fosse a altura de um cilindro (seção 5 - Tronco de prisma triangular e tronco de cilindro)."
    )
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 7: os gigantes de Galileu. Um animal é ampliado 10 vezes em todas
# as direções, mantendo as proporções. Por quanto ficam multiplicados:
# (a) o seu peso;
# (b) a área da seção dos seus ossos;
# (c) a pressão sobre os ossos (peso dividido pela área)?
peso_ex7 = ...
area_ossos_ex7 = ...
pressao_ex7 = ...

print(peso_ex7, area_ossos_ex7, pressao_ex7)

In [ ]:
# Verificação
respostas_ex7 = [("a", peso_ex7, 1000), ("b", area_ossos_ex7, 100), ("c", pressao_ex7, 10)]
for item, resposta, certa in respostas_ex7:
    assert resposta is not ... and abs(resposta - certa) <= 0.001, (
        f"Tente de novo, revise: no item ({item}), o peso acompanha o volume (k³) e a "
        "área da seção acompanha k² (seção 1 - Sólidos semelhantes)."
    )
print("Certo!")

In [ ]:
# Exercício Desafio 8: o balde de pipoca. Um cinema vende pipoca em baldes
# semelhantes, com a forma de tronco de cone. O balde pequeno tem 10 cm de altura,
# 4 cm de raio na boca e 3 cm no fundo, e custa R$ 10,00. O grande tem 15 cm de
# altura (mesmo formato) e custa R$ 25,00.
# (a) Qual é o volume do balde grande, em cm³?
# (b) Qual é o mais vantajoso (menor preço por cm³)? Responda "pequeno" ou "grande".
volume_grande_ex8 = ...
mais_vantajoso_ex8 = ...

print(volume_grande_ex8, mais_vantajoso_ex8)

In [ ]:
# Verificação
assert volume_grande_ex8 is not ... and abs(
    volume_grande_ex8 - 370 * math.pi / 3 * 1.5**3
) <= 1, (
    "Tente de novo, revise: no item (a), os baldes são semelhantes com k = 15/10 = 1,5; "
    "o volume fica multiplicado por k³ (seção 1 - Sólidos semelhantes)."
)
assert mais_vantajoso_ex8 == "grande", (
    "Tente de novo, revise: no item (b), compare preço / volume dos dois baldes; o "
    "volume do pequeno é πh/3 · (R² + Rr + r²) (seção 4 - Tronco de cone de bases paralelas)."
)
print("Certo!")
print("O grande custa 2,5 vezes mais, mas tem 3,375 vezes mais pipoca!")

## Conexões

**Isso depende de:**
- [`0602a_paralelismo.ipynb`](0602a_paralelismo.ipynb) (Teorema de Tales no espaço: as arestas laterais são divididas na mesma razão pelo plano paralelo à base)
- [`0609a_volume_e_principio_de_cavalieri.ipynb`](0609a_volume_e_principio_de_cavalieri.ipynb) (os cubinhos de volume; dobrar as três dimensões multiplica o volume por 8)
- [`0610a_piramide.ipynb`](0610a_piramide.ipynb), [`0611a_cilindro.ipynb`](0611a_cilindro.ipynb) e [`0612a_cone.ipynb`](0612a_cone.ipynb) (volumes de pirâmide, cilindro e cone; a decomposição do prisma triangular em três pirâmides; a planificação do cone)
- [`0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb`](../04_geometria_plana/0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb), [`0415a_semelhanca_de_triangulos_e_potencia_de_ponto.ipynb`](../04_geometria_plana/0415a_semelhanca_de_triangulos_e_potencia_de_ponto.ipynb) e [`0419a_equivalencia_plana.ipynb`](../04_geometria_plana/0419a_equivalencia_plana.ipynb) (trapézio, semelhança e razão entre áreas de figuras semelhantes)

**Isso será usado em:**
- [`0616a_inscricao_e_circunscricao_corpos_redondos.ipynb`](0616a_inscricao_e_circunscricao_corpos_redondos.ipynb) (esfera inscrita num tronco de cone)
- 📌 Superfícies e sólidos de revolução (o tronco de cone gerado por um trapézio girando)
- 📌 Superfícies e sólidos esféricos